In [ ]:
# @title 1. Install Libraries
# Install the specific GeoCLIP model and Hugging Face tools
!pip install geoclip
!pip install datasets
!pip install geopy
!pip install folium mapclassify
!pip install beautifulsoup4

print("Installation complete. Please restart the session if prompted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 18.5 MB/s eta 0:00:00
Installation complete. Please restart the session if prompted.


In [ ]:
# @title 2. Load GeoCLIP Model
import torch
from geoclip import GeoCLIP
from PIL import Image
import requests
from io import BytesIO

# Load the model
# We move it to GPU if available for faster prediction
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = GeoCLIP()
model.to(device)
print("GeoCLIP Model loaded successfully!")

Using device: cuda


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

GeoCLIP Model loaded successfully!


In [ ]:
# @title 2. Download Raw Google Landmarks Data
import os
import pandas as pd
import tarfile

# 1. Download Metadata (The Labels)
if not os.path.exists('train.csv'):
    print("Downloading metadata...")
    !wget -q https://s3.amazonaws.com/google-landmark/metadata/train.csv
    !wget -q https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv

# 2. Download Images (Just one chunk - approx 1GB)
# The full dataset is 500 chunks (TB of data), we only take the first one.
if not os.path.exists('images_000.tar'):
    print("Downloading images (this may take 2-3 minutes)...")
    !wget -q https://s3.amazonaws.com/google-landmark/train/images_000.tar

# 3. Extract Images
if not os.path.exists('train/0'):
    print("Extracting images...")
    with tarfile.open('images_000.tar', 'r') as tar:
        tar.extractall()
    print("Extraction complete!")
else:
    print("Images already extracted.")

Extracting images...


/tmp/ipython-input-952625589.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction complete!


In [ ]:
# @title 3. Build the GPS Extractor (Robust Version)
import requests
from bs4 import BeautifulSoup
import re

def get_gps_from_wikimedia(url):
    """
    Visits the Wikimedia URL and extracts GPS from the 'GeoHack' link.
    Target format: params=52.9345_N_1.1545_E_globe:Earth_
    """
    try:
        # 1. Fetch the page
        headers = {'User-Agent': 'Mozilla/5.0 (StudentProject; Italy)'}
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"Error fetching {url}: Status {response.status_code}")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # 2. Find the link that goes to 'geohack.toolforge.org'
        # This is safer than looking for specific classes which might change
        geohack_link = soup.find('a', href=lambda href: href and "geohack.toolforge.org" in href)

        if geohack_link:
            href = geohack_link['href']

            # 3. Use Regex to extract the coordinates from the URL parameters
            # Looking for pattern like: params=52.9345_N_1.1545_E
            match = re.search(r'params=([\d\.]+)_([NS])_([\d\.]+)_([EW])', href)

            if match:
                lat = float(match.group(1))
                lat_dir = match.group(2)
                lon = float(match.group(3))
                lon_dir = match.group(4)

                # Adjust for South / West
                if lat_dir == 'S':
                    lat = -lat
                if lon_dir == 'W':
                    lon = -lon

                return lat, lon

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

    return None

In [ ]:
# --- TEST BLOCK ---
# Let's test it immediately with the example you found
test_url = "https://commons.wikimedia.org/wiki/Category:North_Norfolk_Railway"
print(f"Testing scraper on: {test_url}")
result = get_gps_from_wikimedia(test_url)

if result:
    print(f"✅ Success! Found coordinates: {result}")
    print(f"   (Latitude: {result[0]}, Longitude: {result[1]})")
else:
    print("❌ Failed to find coordinates.")

Testing scraper on: https://commons.wikimedia.org/wiki/Category:North_Norfolk_Railway
✅ Success! Found coordinates: (52.9345, 1.1545)
   (Latitude: 52.9345, Longitude: 1.1545)


In [ ]:
# @title 4. Create the Benchmark Dataset
import random
import time


# Load the CSVs
df_labels = pd.read_csv('train.csv')
df_categories = pd.read_csv('train_label_to_category.csv')

# Merge them so we have Image ID -> Landmark ID -> Wiki URL
full_df = pd.merge(df_labels, df_categories, on='landmark_id')

# We only want images that we actually downloaded (IDs starting with '000')
# The tar file 'images_000' mostly contains IDs starting with '00'
# Let's find files that actually exist on disk
import glob
found_files = glob.glob("0/*/*/*.jpg") # The extraction structure is 0/a/b/id.jpg

print(f"Found {len(found_files)} images on disk.")

# Select a few random images to test
selected_files = random.sample(found_files, 100) # Testing 5 images

benchmark_data = []

print("Building ground truth for selected images...")
for file_path in selected_files:
    # Extract ID from filename (remove path and extension)
    image_id = os.path.splitext(os.path.basename(file_path))[0]

    # Find row in dataframe
    row = full_df[full_df['id'] == image_id]

    if not row.empty:
        wiki_url = row.iloc[0]['category']
        print(f"Scraping GPS for: {wiki_url}...")

        real_gps = get_gps_from_wikimedia(wiki_url)

        if real_gps:
            benchmark_data.append({
                "image_path": file_path,
                "wiki_url": wiki_url,
                "true_loc": real_gps
            })
            print(f"  -> Found: {real_gps}")
        else:
            print("  -> No GPS found on page.")

        time.sleep(1) # Be polite to Wikipedia servers

print(f"\nReady to test on {len(benchmark_data)} images.")

Found 8266 images on disk.
Building ground truth for selected images...
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Albarine...
  -> Found: (45.96479, 5.25772)
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Venta_Rapid...
  -> Found: (56.968055555556, 21.978888888889)
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:St_Stevenskerk_(Nijmegen)...
  -> Found: (51.847778, 5.8625)
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Toronto_Harbour...
  -> No GPS found on page.
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Eerste_Kamergebouw...
  -> Found: (52.0796, 4.31207)
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Notre_Dame_de_Lorette...
  -> No GPS found on page.
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Ny_Kirke...
  -> Found: (55.13944444, 14.76916667)
Scraping GPS for: http://commons.wikimedia.org/wiki/Category:Lantau_Island...
  -> Found: (22.270555555556, 113.95277777778)
Scrapin

In [ ]:
# @title 5. Run GeoCLIP Evaluation
from geoclip import GeoCLIP
from geopy.distance import geodesic
from PIL import Image

# Load Model
model = GeoCLIP()
model.to(device)

print("-" * 50)
print(" STARTING EVALUATION ")
print("-" * 50)

for item in benchmark_data:
    # Load Image
    img = Image.open(item['image_path'])

    # Predict
    top_pred_gps, _ = model.predict(item['image_path'], top_k=1)
    pred_loc = (top_pred_gps[0][0], top_pred_gps[0][1])

    # Calculate Error
    true_loc = item['true_loc']
    error_km = geodesic(true_loc, pred_loc).km

    item['pred_loc'] = pred_loc
    item['error'] = error_km

    print(f"Wiki Category: {item['wiki_url'].split(':')[-1]}")
    print(f"TRUE: {true_loc}")
    print(f"PRED: {pred_loc}")
    print(f"ERROR: {error_km:.2f} km")
    print("-" * 30)

--------------------------------------------------
 STARTING EVALUATION 
--------------------------------------------------
Wiki Category: Albarine
TRUE: (45.96479, 5.25772)
PRED: (tensor(43.9491), tensor(6.8098))
ERROR: 255.28 km
------------------------------
Wiki Category: Venta_Rapid
TRUE: (56.968055555556, 21.978888888889)
PRED: (tensor(52.8362), tensor(-6.2011))
ERROR: 1850.76 km
------------------------------
Wiki Category: St_Stevenskerk_(Nijmegen)
TRUE: (51.847778, 5.8625)
PRED: (tensor(53.4052), tensor(6.6844))
ERROR: 182.03 km
------------------------------
Wiki Category: Eerste_Kamergebouw
TRUE: (52.0796, 4.31207)
PRED: (tensor(41.0792), tensor(14.3276))
ERROR: 1440.82 km
------------------------------
Wiki Category: Ny_Kirke
TRUE: (55.13944444, 14.76916667)
PRED: (tensor(54.8881), tensor(10.4098))
ERROR: 280.23 km
------------------------------
Wiki Category: Lantau_Island
TRUE: (22.270555555556, 113.95277777778)
PRED: (tensor(44.3039), tensor(9.2098))
ERROR: 9410.28 km
--